In [1]:
import pandas as pd
import implicit
from sklearn.model_selection import train_test_split
import scipy.sparse as sparse
from pandas.api.types import CategoricalDtype 
import numpy as np
import warnings
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
from catboost import CatBoostClassifier, Pool

## Задание 1. Создание user-item-матрицы и разбиение данных на тест и контроль результатов

In [3]:
# Загрузка данных
ratings = pd.read_csv('ratings.csv')
ratings

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3
...,...,...,...
5976474,49925,510,5
5976475,49925,528,4
5976476,49925,722,4
5976477,49925,949,5


In [4]:
# Преобразование рейтингов в бинарные оценки
# Будем считать, если оценка больше 3, то книга понравилась
ratings['rating'] = ratings['rating'].apply(lambda x: 1 if x > 3 else 0)

# Пронумеруем для каждого пользователя его прочитанные книги согласно расположению книг по порядку
ratings['book_order'] = ratings.groupby('user_id')['book_id'].rank(method='first')

# Переведем номера прочитанных книг в доли по формуле «порядковый номер / общее количество прочитанных пользователем книг»
ratings['book_order_ratio'] = ratings['book_order'] / ratings.groupby('user_id')['book_id'].transform('count')

ratings

,user_id,book_id,rating,book_order,book_order_ratio
0,1,258,1,63.0,0.538462
1,2,4081,1,54.0,0.830769
2,2,260,1,30.0,0.461538
3,2,9296,1,63.0,0.969231
4,2,2318,0,48.0,0.738462
...,...,...,...,...,...
5976474,49925,510,1,41.0,0.303704
5976475,49925,528,1,42.0,0.311111
5976476,49925,722,1,45.0,0.333333
5976477,49925,949,1,51.0,0.377778


In [5]:
# Разделите данные на обучение и контроль
train, test = train_test_split(ratings, train_size=0.8, random_state=42, shuffle=False)

In [6]:
# Построение user-item-матрицы
train_user_index = train.user_id.unique()
train_book_index = train.book_id.unique()
test_user_index = test.user_id.unique()
test_book_index = test.book_id.unique()

rows = train['user_id'].astype(pd.CategoricalDtype(categories=train_user_index)).cat.codes
cols = train['book_id'].astype(pd.CategoricalDtype(categories=train_book_index)).cat.codes
train_matrix = sparse.csr_matrix((train.rating, (rows, cols)), shape=(len(train_user_index), len(train_book_index)))

rows = test['user_id'].astype(pd.CategoricalDtype(categories=test_user_index)).cat.codes
cols = test['book_id'].astype(pd.CategoricalDtype(categories=test_book_index)).cat.codes
test_matrix = sparse.csr_matrix((test.rating, (rows, cols)), shape=(len(test_user_index), len(test_book_index)))


In [7]:
train_matrix = train_matrix.toarray()
test_matrix = test_matrix.toarray()

In [8]:
# Проверка размеров матриц
if train_matrix.shape[0] == len(train_user_index):
    print(f"Количество строк в обучающей матрице ({train_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(train_user_index)})")
else:
    print(f"Ошибка! Количество строк в обучающей матрице ({train_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(train_user_index)})")

if train_matrix.shape[1] == len(train_book_index):
    print(f"Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(train_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(train_book_index)})")

if test_matrix.shape[0] == len(test_user_index):
    print(f"Количество строк в тестовой матрице ({test_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(test_user_index)})")
else:
    print(f"Ошибка! Количество строк в тестовой матрице ({test_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(test_user_index)})")

if test_matrix.shape[1] == len(test_book_index):
    print(f"Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(test_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(test_book_index)})")


Количество строк в обучающей матрице (48769) РАВНО Количеству уникальных пользователей (48769)
Количество столбцов в обучающей матрице (9693) РАВНО Количеству уникальных книг (9693)
Количество строк в тестовой матрице (32281) РАВНО Количеству уникальных пользователей (32281)
Количество столбцов в тестовой матрице (9999) РАВНО Количеству уникальных книг (9999)


## Задание 2. Применение метода матричной факторизации и сбор признаков для контентной модели

In [9]:
# Для удобства возьмем 2-го пользователя из списка
user_index = 2

In [10]:
def ap_at_k(recommendations, true_positives, k=10):
    """
    Подсчитывает метрику AP@K.

    Args:
        recommendations: Список рекомендаций (book_id).
        true_positives: Список книг, положительно оценённых пользователем.
        k: Количество рекомендаций, которые учитываются при подсчёте метрики.

    Returns:
        Значение метрики AP@K.
    """
    
    # Отрезаем список рекомендаций до k элементов
    recommendations = recommendations[:k]
    
    # Создаем список бинарных меток (1 - книга в рекомендациях есть в списке положительно оцененных, 0 - нет)
    y_true = [1 if book_ind in true_positives else 0 for book_ind in recommendations]
    
    # Используем sklearn.metrics.average_precision_score для подсчета AP@K
    ap = average_precision_score(y_true, [1]*len(y_true)) 
    
    return ap

In [11]:
random_users_inds = np.random.choice(len(train_user_index), size=50, replace=False)

In [12]:
from tqdm import tqdm 

In [13]:
user_id = train_user_index[user_index]

In [14]:
# Создание модели ALS
als_model = implicit.als.AlternatingLeastSquares(factors=64, iterations=30)

als_model.fit(sparse.csr_matrix(train_matrix), show_progress=True)


C:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 24 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

In [15]:
# Расчет mAP@10
als_map10 = []
for rand_user_ind in tqdm(random_users_inds):  # Используем tqdm для прогрессбара
    user_id = train_user_index[rand_user_ind]
    true_positives = test[test.user_id == user_id][test.rating == 1].book_id.tolist()

    if true_positives:
        recommendations = als_model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=20)
        recommended_book_ids = train_book_index[recommendations[0]]
        ap10 = ap_at_k(recommended_book_ids, true_positives, k=10)
        als_map10.append(ap10)

100%|██████████████████████████████████████████| 50/50 [01:30<00:00,  1.82s/it]


In [16]:
mean_als_ap_at_10 = np.mean(als_map10)
print(f"Среднее значение AP@10 для ALS: {mean_als_ap_at_10}")


Среднее значение AP@10 для ALS: 0.05151515151515152


### Задание 3. Применение метода матричной факторизации и улучшение параметров алгоритма факторизации

In [17]:
train_df = train.copy()
test_df = test.copy()

books = pd.read_csv('books.csv')
genres_df = pd.read_json('goodreads_book_genres_initial.json', lines=True)

genres_df = genres_df[genres_df.book_id.isin(books.goodreads_book_id)]
genres_df.columns = ['book_id', 'genres_dict']

all_genres = set()
for dict_genre in genres_df.genres_dict:
    for elem in list(dict_genre.keys()):
        all_genres.add(elem)

for genre in all_genres:
    genres_df[genre] = 0

def simple_one_hot(genre_dict, genre):
    if genre in genre_dict:
        return 1
    return 0

for genre in all_genres:
    genres_df[genre] = genres_df.apply(lambda df: simple_one_hot(df['genres_dict'], genre), axis=1)

train_df = train_df.merge(books[['book_id', 'goodreads_book_id']], left_on='book_id', right_on='book_id', how='left')
train_df = train_df.merge(genres_df, left_on='goodreads_book_id', right_on='book_id', how='left')
test_df = test_df.merge(books[['book_id', 'goodreads_book_id']], left_on='book_id', right_on='book_id', how='left')
test_df = test_df.merge(genres_df, left_on='goodreads_book_id', right_on='book_id', how='left')

users_profiles = train_df.groupby('user_id')[list(all_genres)].sum()
users_profiles.columns = ['user_'+name for name in list(users_profiles)]

train_df.columns = ['book_'+item if item in all_genres else item for item in list(train_df)]
test_df.columns = ['book_'+item if item in all_genres else item for item in list(test_df)]

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=100, stop_words=stopwords.words('english'))
X = vectorizer.fit_transform(books.title)

train_df = train_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')
test_df = test_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')

tf_idf_df = pd.DataFrame(X.toarray(), columns=vectorizer.vocabulary_)
tf_idf_df.columns = ['tf_' + item for item in tf_idf_df.columns] 

books = pd.concat([books, tf_idf_df], axis=1)

train_df = train_df.merge(books[list(tf_idf_df)+['book_id']], left_on='book_id_x', right_on='book_id')
test_df = test_df.merge(books[list(tf_idf_df)+['book_id']], left_on='book_id_x', right_on='book_id')

# 1. Признак для пользователя (средний рейтинг)
user_avg_ratings = ratings.groupby('user_id')['rating'].mean().reset_index()
user_avg_ratings = user_avg_ratings.rename(columns={'rating': 'user_avg_rating'})
train_df = train_df.merge(user_avg_ratings, on='user_id', how='left')
test_df = test_df.merge(user_avg_ratings, on='user_id', how='left')
users_profiles = users_profiles.merge(user_avg_ratings, on='user_id', how='left')

# 2. Признак для книги (популярность)
books['book_popularity'] = np.log(books['ratings_count'] + 1) # +1 чтобы избежать логарифма от нуля
train_df = train_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')
test_df = test_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')

books = books.merge(genres_df, on='book_id', how='left')
books.columns = ['book_'+item if item in all_genres else item for item in list(books)]

In [18]:
cols_for_using = [
 'book_mystery, thriller, crime',
 'book_non-fiction',
 'book_romance',
 'book_fantasy, paranormal',
 'book_poetry',
 'book_fiction',
 'book_young-adult',
 'book_history, historical fiction, biography',
 'book_comics, graphic',
 'book_children',
 'user_mystery, thriller, crime',
 'user_non-fiction',
 'user_romance',
 'user_fantasy, paranormal',
 'user_poetry',
 'user_fiction',
 'user_young-adult',
 'user_history, historical fiction, biography',
 'user_comics, graphic',
 'user_children', 
 'user_avg_rating',
 'book_popularity']+list(tf_idf_df)

In [19]:
# Обучение модели
CB_model = CatBoostClassifier(
                            iterations=1000,
                            task_type="GPU",
                            use_best_model=True,
                            eval_metric='Accuracy',
                            loss_function='MultiClass',
                            early_stopping_rounds=10,
                            custom_metric='Precision'  
                        )

CB_model.fit(train_df[cols_for_using], 
              train_df['rating'], 
              eval_set=(test_df[cols_for_using], test_df['rating']),
              verbose=False,
              plot=True) 

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [20]:
# --- Обучение ALS ---

In [21]:
# --- Получение рекомендаций от ALS ---
als_recommendations = []
for rand_user_ind in tqdm(random_users_inds):
    user_id = train_user_index[rand_user_ind]  # Получаем user_id по индексу из train_user_index
    recommendations = als_model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=30)  # Получаем рекомендации
    als_recommended_book_ids = train_book_index[recommendations[0]]  # Преобразуем индексы книг в реальные ID
    als_recommendations.append(als_recommended_book_ids)  # Сохраняем рекомендации в список

100%|██████████████████████████████████████████| 50/50 [02:17<00:00,  2.74s/it]


In [22]:
common_columns_users = list(set(train_df.columns) & set(users_profiles.columns))
common_columns_users

['user_fantasy, paranormal',
 'user_history, historical fiction, biography',
 'user_young-adult',
 'user_non-fiction',
 'user_fiction',
 'user_avg_rating',
 'user_poetry',
 'user_id',
 'user_mystery, thriller, crime',
 'user_romance',
 'user_children',
 'user_comics, graphic']

In [23]:
train_df

,user_id,book_id_x,rating,book_order,book_order_ratio,goodreads_book_id,book_id_y,genres_dict,book_romance,"book_comics, graphic",...,tf_rising,tf_hunter,tf_woman,tf_12,tf_13,tf_summer,tf_star,book_id,user_avg_rating,book_popularity
0,1,258,1,63.0,0.538462,1232,1232.0,"{'fiction': 8627, 'history, historical fiction...",1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,258,0.521368,12.482514
1,2,4081,1,54.0,0.830769,231,231.0,"{'fiction': 707, 'young-adult': 16, 'romance': 4}",1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4081,0.830769,9.867549
2,2,260,1,30.0,0.461538,4865,4865.0,{'non-fiction': 3579},0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,260,0.830769,12.551873
3,2,9296,1,63.0,0.969231,4887,4887.0,"{'non-fiction': 367, 'children': 5}",0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9296,0.830769,9.165761
4,2,2318,0,48.0,0.738462,998,998.0,{'non-fiction': 896},0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2318,0.830769,10.690535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4781178,50331,1108,1,61.0,0.586538,46306,46306.0,"{'children': 1353, 'fiction': 483, 'young-adul...",1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1108,0.817308,11.418615
4781179,50331,3595,1,91.0,0.875000,23522,23522.0,"{'non-fiction': 923, 'fiction': 396, 'history,...",0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3595,0.817308,10.335432
4781180,43658,2486,0,76.0,0.745098,22522805,22522805.0,"{'fantasy, paranormal': 1637, 'fiction': 1995,...",0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2486,0.921569,10.350478
4781181,50331,1320,1,65.0,0.625000,3586,3586.0,"{'mystery, thriller, crime': 621, 'fiction': 3...",0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1320,0.817308,11.110326


In [24]:
common_columns_books = list(set(train_df.columns) & set(books.columns))
common_columns_books

['tf_school',
 'tf_midnight',
 'tf_complete',
 'tf_magic',
 'tf_13',
 'tf_black',
 'tf_glass',
 'tf_things',
 'tf_rising',
 'tf_history',
 'tf_earth',
 'book_mystery, thriller, crime',
 'tf_moon',
 'book_fiction',
 'tf_one',
 'goodreads_book_id',
 'tf_new',
 'tf_12',
 'tf_first',
 'tf_book',
 'tf_ice',
 'tf_way',
 'tf_princess',
 'tf_boy',
 'tf_volume',
 'tf_blue',
 'tf_game',
 'tf_woman',
 'tf_power',
 'tf_world',
 'tf_little',
 'tf_last',
 'book_history, historical fiction, biography',
 'tf_great',
 'tf_club',
 'tf_war',
 'tf_city',
 'tf_universe',
 'tf_time',
 'tf_house',
 'tf_children',
 'tf_alex',
 'tf_lost',
 'book_poetry',
 'tf_women',
 'tf_hunter',
 'book_young-adult',
 'tf_heart',
 'tf_bosch',
 'tf_saga',
 'book_comics, graphic',
 'tf_tale',
 'tf_day',
 'book_popularity',
 'tf_dead',
 'tf_family',
 'tf_fallen',
 'book_children',
 'tf_darkness',
 'tf_story',
 'tf_fire',
 'tf_love',
 'tf_sea',
 'tf_big',
 'genres_dict',
 'tf_shadow',
 'tf_harry',
 'tf_girls',
 'tf_trilogy',
 'tf

In [26]:
hybrid_map10 = []
for i, rand_user_ind in enumerate(tqdm(random_users_inds)):
    user_id = train_user_index[rand_user_ind]  # Получаем user_id по индексу из train_user_index
    true_positives = test_df[test_df.user_id == user_id][test_df.rating == 1].book_id.tolist()

    if true_positives:
        # Ранжирование с помощью CatBoost
       # Проверка, что user_id существует в users_profiles.index
        if user_id in users_profiles.index:
            user_features = users_profiles[common_columns_users].loc[user_id] # Получение признаков пользователя
        else:
            print(f"Ошибка: Пользователь {user_id} отсутствует в тестовой выборке ")  
        candidate_features = books[books.book_id.isin(als_recommendations[i])][common_columns_books]  # Используем als_recommendations[i]

        
        # Создание features DataFrame
        features = pd.concat([user_features.to_frame().T] * len(candidate_features), ignore_index=True).join(
            candidate_features, lsuffix='user', rsuffix='book'
        )

        # Проверка порядка признаков и перестановка, если нужно
        features = features[train_df[cols_for_using].columns]  # Устанавливаем порядок признаков, соответствующий train_df

        # Проверка на наличие пропущенных значений и заполнение их нулями
        features.fillna(0, inplace=True)

        predictions = CB_model.predict_proba(
            features
        )[:, 1]  # Получение вероятностей для класса 1 (положительный рейтинг)
        top_recommendations = als_recommendations[i][np.argsort(predictions)[::-1][:10]]  # Выбор топ-10 книг по ранжированию

        # Расчет mAP@10
        ap10 = ap_at_k(top_recommendations, true_positives, k=10)
        hybrid_map10.append(ap10)

mean_hybrid_ap_at_10 = np.mean(hybrid_map10)
print(f"Среднее значение AP@10 для гибридной модели: {mean_hybrid_ap_at_10}")

 20%|████████▍                                 | 10/50 [00:11<00:38,  1.03it/s]

Ошибка: Пользователь 49741 отсутствует в тестовой выборке 


 40%|████████████████▊                         | 20/50 [00:23<00:31,  1.06s/it]

Ошибка: Пользователь 49047 отсутствует в тестовой выборке 


100%|██████████████████████████████████████████| 50/50 [00:52<00:00,  1.04s/it]

Среднее значение AP@10 для гибридной модели: 0.04242424242424243
